In [ ]:
import sys, platform, importlib
from google.colab import auth

auth.authenticate_user(project_id="sjsu-cs131-team04-sec1")

PROJECT_ID = "sjsu-cs131-team04-sec1"
BUCKET_NAME = "datascrapers"
BUCKET_URI = f"gs://{BUCKET_NAME}"


print("Python:", sys.version)
print("Platform:", platform.platform())

for pkg in ["pyspark", "pandas", "numpy", "pyarrow"]:
    try:
        m = importlib.import_module(pkg)
        print(pkg, getattr(m, "__version__", "unknown"))
    except Exception:
        print(pkg, "not installed")

try:
    import pyspark
except ImportError:
    !pip install -q pyspark
    import pyspark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("team-colab-spark")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

print("Spark version:", spark.version)
spark.range(5).show()

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
pyspark 4.0.2
pandas 2.2.2
numpy 2.0.2
pyarrow 18.1.0
Spark version: 4.0.2
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [ ]:
from google.colab import auth
auth.authenticate_user(project_id="sjsu-cs131-team04-sec1")

!gcloud config set project sjsu-cs131-team04-sec1
!gsutil ls gs://datascrapers-team04/

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
You are attempting to access protected data with no configured
credentials. Please visit https://cloud.google.com/console#/project
and sign up for an account, and then run the "gcloud auth login"
command to configure gsutil to use these credentials.


### Authenticate `gcloud` for `gsutil` access

The previous `gsutil` commands failed because the shell environment was not authenticated. Please run the cell below and follow the prompts in the new browser window to authenticate your Google Cloud SDK. Once authenticated, you can re-run the cells that attempt to copy or list files from the bucket.

Two PySpark tables

One Top-N Table with reviews per product using asin (product ID) and title (product name).

One Ratio Table with verified purchase by rating using rating and verified_purchase.

In [ ]:
!gsutil cp gs://datascrapers/Beauty_and_Personal_Care.jsonl.gz .

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://datascrapers/Beauty_and_Personal_Care.jsonl.gz...
==> NOTE: You are downloading one or more large file(s), which would
run significantly faster if you enabled sliced object downloads. This
feature is enabled by default but requires that compiled crcmod be
installed (see "gsutil help crcmod").

- [1 files][  2.8 GiB/  2.8 GiB]   78.8 MiB/s                                   
Operation completed over 1 objects/2.8 GiB.                                      


In [ ]:
from google.colab import auth
auth.authenticate_user(project_id="sjsu-cs131-team04-sec1")

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, first, sum, when, round
import matplotlib.pyplot as plt

spark = SparkSession.builder \
    .appName("PA5") \
    .getOrCreate()

# reading the dataset file
df = spark.read.json("Beauty_and_Personal_Care.jsonl.gz")

clean_df = df.select(
    col("parent_asin").alias("asin"),
    col("title"),
    col("rating").cast("int").alias("rating"),
    col("verified_purchase")
).filter(
    col("parent_asin").isNotNull() &
    col("title").isNotNull() &
    col("rating").isNotNull() &
    col("verified_purchase").isNotNull()
)

print("\n")

# the first table
top_n_table = clean_df.groupBy("asin").agg(
    first("title").alias("title"),
    count("*").alias("review_count")
).orderBy(col("review_count").desc())

print("\n")
print("Top-N Table")
top_n_table.show()
print("\n")

# second table
ratio_table = clean_df.groupBy("rating").agg(
    count("*").alias("total_reviews"),
    sum(
        when(col("verified_purchase") == True, 1)
        .otherwise(0)
    ).alias("verified_reviews")
).withColumn(
    "verified_ratio",
    round(col("verified_reviews") / col("total_reviews"), 4)
).orderBy("rating")

print("\n")
print("Ratio Table")
print("\n")

ratio_table.show()

print("\n")

# bar graph
rating_counts = clean_df.groupBy("rating").count().orderBy("rating")
pdf = rating_counts.toPandas()

plt.bar(pdf["rating"], pdf["count"])
plt.xlabel("Rating")
plt.ylabel("Number of Reviews")
plt.title("Review Count by Rating")
plt.show()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
ERROR:py4j.clientserver:Exception occurred while shutting down connection
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver

KeyboardInterrupt: 

# Skinny Tables


In [ ]:
from pyspark.sql.functions import col, count, round, sum as _sum
from pyspark.sql.window import Window


skinny_df = clean_df.groupBy("verified_purchase", "rating").agg(
    count("*").alias("review_count")
)


window_spec = Window.partitionBy()

skinny_table = skinny_df.withColumn(
    "total", _sum("review_count").over(window_spec)
).withColumn(
    "pct_of_total", round(col("review_count") / col("total") * 100, 2)
).drop("total").orderBy("verified_purchase", "rating")

skinny_table.show()


+-----------------+------+------------+------------+
|verified_purchase|rating|review_count|pct_of_total|
+-----------------+------+------------+------------+
|            false|     1|      222293|        0.93|
|            false|     2|      104441|        0.44|
|            false|     3|      161692|        0.68|
|            false|     4|      350681|        1.47|
|            false|     5|     1286914|        5.38|
|             true|     1|     2618585|       10.95|
|             true|     2|     1170590|         4.9|
|             true|     3|     1554129|         6.5|
|             true|     4|     2265763|        9.48|
|             true|     5|    14176302|       59.29|
+-----------------+------+------------+------------+

